# Day 47 · 2 — HDFS ingestion, latest orders and Iceberg SCD1

Run after any capture in notebook 1. All cells can be rerun. Each new immutable batch is
copied once to HDFS, then the small full raw history is read again. This deliberately trades
scale for transparent, replayable code. Iceberg MERGE makes target writes idempotent.
No Hive Metastore, dependency download, Kafka or Snowflake is involved.

**Optional fresh load:** before starting Spark, run `bash reset_hdfs.sh` in a terminal.
The script's `DB` must match this notebook. It deletes that database's directories under
`/bronze` and `/warehouse` if present. Restart
any existing Spark kernels after a reset. This notebook then copies the retained local
batches again; notebook 3 rebuilds SCD2 history. Do not reset between incremental changes.

In [ ]:
import os
import re
from pathlib import Path

DB = "cdc_scd_db"  # Use the same database name in all three notebooks.
assert re.fullmatch(r"[a-zA-Z][a-zA-Z0-9_]{0,63}", DB)
LAB_DIR = Path.cwd() / "lab_data" / DB
LAB_DIR.mkdir(parents=True, exist_ok=True)
BRONZE = f"hdfs:///bronze/{DB}"
WAREHOUSE = f"hdfs:///warehouse/{DB}"
print("MySQL database:", DB)

## Start Spark 3.5 with a compatible Iceberg runtime

The configuration derives `/bronze/<DB>` and `/warehouse/<DB>` from the database name.
Use an HDFS account with write access to these locations.
Use a kernel configured with Spark 3.5, Java, Hadoop configuration and a compatible Iceberg
runtime JAR. The Hadoop catalog stores metadata and Parquet files directly on HDFS.
No dependencies are downloaded by this notebook.

In [ ]:
import sys
import subprocess

# Use the Spark, Java and Hadoop installation configured for your notebook kernel.
os.environ["PYSPARK_PYTHON"] = sys.executable
import pyspark
assert pyspark.__version__.startswith("3.5."), "Use a Spark 3.5 kernel with a matching Iceberg runtime."
from pyspark.sql import SparkSession, Window, functions as F

active = SparkSession.getActiveSession()
if active is not None:
    assert active.conf.get("spark.sql.catalog.lab.warehouse", "") == WAREHOUSE, "Restart the kernel when switching DB or Spark configuration."
    assert "IcebergSparkSessionExtensions" in active.conf.get("spark.sql.extensions", ""), "Restart the kernel to enable Iceberg."

spark = (SparkSession.builder.master("local[2]").appName(DB)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lab", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lab.type", "hadoop")
    .config("spark.sql.catalog.lab.warehouse", WAREHOUSE)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.sql("CREATE NAMESPACE IF NOT EXISTS lab.ecommerce")
print("Spark:", spark.version)

## Upload only unpublished batches

The whole directory is staged under a hidden name and renamed after upload, so readers
see completed batches. HDFS paths are scoped by DB. Existing published files are not replaced.

In [ ]:
import uuid
import shutil
HDFS = shutil.which("hdfs")
assert HDFS, "Make the Hadoop hdfs command available in your kernel environment."

def hdfs(*args):
    return subprocess.run([HDFS, "dfs", *args], check=True, capture_output=True, text=True)

hdfs("-mkdir", "-p", BRONZE)
batches = sorted((LAB_DIR / "batches").glob("batch_*"))
assert batches, "Run notebook 1: Change 0 and capture first."
for batch in batches:
    assert (batch / "manifest.json").exists()
    destination = BRONZE + "/" + batch.name
    exists = subprocess.run([HDFS, "dfs", "-test", "-e", destination], capture_output=True)
    if exists.returncode == 0:
        continue
    assert exists.returncode == 1, exists.stderr.decode()
    temporary = BRONZE + "/.upload_" + uuid.uuid4().hex
    hdfs("-put", str(batch), temporary)
    hdfs("-mv", temporary, destination)
    print("Published", destination)
print(hdfs("-ls", BRONZE).stdout)

## Bronze: inspect CDC images and the complete order-event history

Use explicit types: money is DECIMAL, identifiers are BIGINT and timestamps are UTC.
`before`/`after` remain in the raw JSON. Flat after/current fields simplify downstream SQL.

In [ ]:
product_image = "STRUCT<product_id: BIGINT, product_name: STRING, category: STRING, color: STRING, list_price: DECIMAL(10,2), updated_at: TIMESTAMP>"
product_schema = ("batch_id BIGINT, sequence BIGINT, cdc_id STRING, operation STRING, product_id BIGINT, "
    "product_name STRING, category STRING, color STRING, list_price DECIMAL(10,2), event_time TIMESTAMP, "
    "binlog_time TIMESTAMP, source_updated_at TIMESTAMP, before " + product_image + ", after " + product_image)
order_schema = ("event_id BIGINT, order_id BIGINT, product_id BIGINT, quantity INT, selling_price DECIMAL(10,2), "
                "status STRING, event_time TIMESTAMP, created_at TIMESTAMP")
raw_products = spark.read.schema(product_schema).json(BRONZE + "/batch_*/product_cdc_*.json")
orders = (spark.read.schema(order_schema).option("header", True).option("mode", "FAILFAST")
    .csv(BRONZE + "/batch_*/orders_*.csv"))
raw_products.orderBy("sequence").show(truncate=False)
orders.orderBy("event_id").show(100, truncate=False)
assert raw_products.filter("cdc_id IS NULL OR product_id IS NULL OR event_time IS NULL").count() == 0
assert orders.filter("event_id IS NULL OR event_time IS NULL").count() == 0
assert raw_products.select("cdc_id").distinct().count() == raw_products.count()
assert orders.select("event_id").distinct().count() == orders.count()

## Silver: flatten and preserve every change

Do not deduplicate products to their latest row here: SCD2 needs intermediate changes,
including several updates to the same product in one captured batch.

In [ ]:
changes = raw_products.select("batch_id", "sequence", "cdc_id", "operation", "product_id",
    "product_name", "category", "color", "list_price", "event_time")
changes.orderBy("sequence").show(100, truncate=False)
changes.createOrReplaceTempView("incoming_changes")
orders.createOrReplaceTempView("incoming_orders")
spark.sql("CREATE TABLE IF NOT EXISTS lab.ecommerce.product_changes USING iceberg AS SELECT * FROM incoming_changes WHERE 1=0")
spark.sql("CREATE TABLE IF NOT EXISTS lab.ecommerce.order_history USING iceberg AS SELECT * FROM incoming_orders WHERE 1=0")
spark.sql("""MERGE INTO lab.ecommerce.product_changes t USING incoming_changes s
    ON t.cdc_id=s.cdc_id WHEN NOT MATCHED THEN INSERT *""")
spark.sql("""MERGE INTO lab.ecommerce.order_history t USING incoming_orders s
    ON t.event_id=s.event_id WHEN NOT MATCHED THEN INSERT *""")

## Resolve current order state by business key

Business time comes first; physical `event_id` breaks ties. Arrival time would incorrectly
turn order 1001 back to PAID after Change 3. A DELIVERED filter loses CANCELLED, PAID and CREATED
orders. Keep all events in `order_history` and exactly one row per order in `latest_orders`.

In [ ]:
history = spark.table("lab.ecommerce.order_history")
window = Window.partitionBy("order_id").orderBy(F.col("event_time").desc(), F.col("event_id").desc())
latest = history.withColumn("rn", F.row_number().over(window)).filter("rn=1").drop("rn")
latest.createOrReplaceTempView("resolved_orders")
spark.sql("CREATE TABLE IF NOT EXISTS lab.ecommerce.latest_orders USING iceberg AS SELECT * FROM resolved_orders WHERE 1=0")
spark.sql("""MERGE INTO lab.ecommerce.latest_orders t USING resolved_orders s
    ON t.order_id=s.order_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
history.groupBy("order_id").count().orderBy("order_id").show(truncate=False)
latest.orderBy("order_id").show(truncate=False)
history.filter("order_id=1001").orderBy("created_at", "event_id").show(truncate=False)
print("All current orders:", latest.count(), "| Delivered only:", latest.filter("status='DELIVERED'").count())

## SCD Type 1: overwrite the current representation

One source row per product is required for MERGE. Resolve the latest CDC row by source
sequence; retain DELETE rows until MERGE so an older INSERT cannot resurrect a deleted product.
This lab replays all immutable changes, so a rerun computes the same current state.

In [ ]:
spark.sql("""CREATE TABLE IF NOT EXISTS lab.ecommerce.dim_product_scd1 (
    product_id BIGINT, product_name STRING, category STRING, color STRING,
    list_price DECIMAL(10,2), updated_at TIMESTAMP) USING iceberg""")
product_window = Window.partitionBy("product_id").orderBy(F.col("sequence").desc())
current_changes = (spark.table("lab.ecommerce.product_changes")
    .withColumn("rn", F.row_number().over(product_window)).filter("rn=1").drop("rn"))
current_changes.createOrReplaceTempView("current_product_changes")
spark.sql("""MERGE INTO lab.ecommerce.dim_product_scd1 t USING current_product_changes s
    ON t.product_id=s.product_id
    WHEN MATCHED AND s.operation='D' THEN DELETE
    WHEN MATCHED AND s.operation<>'D' THEN UPDATE SET
        t.product_name=s.product_name, t.category=s.category, t.color=s.color,
        t.list_price=s.list_price, t.updated_at=s.event_time
    WHEN NOT MATCHED AND s.operation<>'D' THEN INSERT
        (product_id,product_name,category,color,list_price,updated_at)
        VALUES(s.product_id,s.product_name,s.category,s.color,s.list_price,s.event_time)""")
spark.table("lab.ecommerce.dim_product_scd1").orderBy("product_id").show(truncate=False)

## Price-period analysis without double-counting lifecycle events

Use actual selling price, not today's list price. The measure below is **booked value for
currently PAID or DELIVERED orders**, not accounting revenue. CANCELLED and CREATED are excluded
only from this measure, never from latest-order resolution. One lifecycle row per order
prevents counting the same order once for CREATED and again for PAID.

In [ ]:
spark.sql("""SELECT selling_price, COUNT(*) AS orders, SUM(quantity) AS quantity,
    SUM(quantity * selling_price) AS booked_value
    FROM lab.ecommerce.latest_orders WHERE status IN ('PAID','DELIVERED')
    GROUP BY selling_price ORDER BY selling_price DESC""").show(truncate=False)
assert latest.count() == latest.select("order_id").distinct().count()
print("Next: run notebook 3 to preserve product history.")